# Budgerigar M0：双控制流与可微发声执行器
独立验证声源/驱动控制、构音/声道控制和连续渲染器；本阶段不接短期记忆或自身反馈。

In [ ]:
#@title 1. 更新项目
REPO_DIR='/content/Budgerigar'
from pathlib import Path
import subprocess,sys,importlib,shutil,datetime
repo=Path(REPO_DIR)
if not (repo/'.git').is_dir():subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
else:
 pull=subprocess.run(['git','-C',REPO_DIR,'pull','--ff-only'],text=True,capture_output=True);print(pull.stdout,pull.stderr)
 if pull.returncode:
  backup=repo.with_name(f'Budgerigar_backup_{datetime.datetime.now():%H%M%S}');shutil.move(str(repo),str(backup));subprocess.run(['git','clone','--depth=1','https://github.com/DoctorAwe/Budgerigar.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',f'{REPO_DIR}[train]'],check=True);sys.path.insert(0,REPO_DIR)
for name in [k for k in list(sys.modules) if k=='budgerigar' or k.startswith('budgerigar.')]:del sys.modules[name]
importlib.invalidate_caches();commit=subprocess.run(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'],capture_output=True,text=True).stdout.strip();print('commit:',commit)

In [ ]:
#@title 2. Drive、数据与流式等价检查
from google.colab import drive
drive.mount('/content/drive');WORK_ROOT=Path('/content/drive/MyDrive/Budgerigar');MANIFEST=WORK_ROOT/'manifests'/'fsdd.jsonl'
import torch,json
if not torch.cuda.is_available():raise RuntimeError('请选择 GPU runtime')
from budgerigar.sensorimotor_vocoder import SensorimotorVocoderConfig,create_sensorimotor_vocoder
config=SensorimotorVocoderConfig();model=create_sensorimotor_vocoder(config).cuda().eval();dummy=torch.randn(2,12,config.tick_samples,device='cuda')
with torch.no_grad():full,_=model(dummy);first,a=model(dummy[:,:5]);second,b=model(dummy[:,5:],a['analysis_state'],a['motor_state'],a['phase'])
difference=float((full-torch.cat([first,second],1)).abs().max());print('parameters:',sum(p.numel() for p in model.parameters()),'stream difference:',difference);assert difference<1e-5

In [ ]:
#@title 3. T4 M0 oracle-source articulation isolation training
MAX_STEPS=500 #@param {type:'integer'}
BATCH_SIZE=8 #@param {type:'integer'}
from budgerigar.train_sensorimotor_vocoder import SensorimotorTrainingConfig,train_sensorimotor_vocoder
RUN_DIR=WORK_ROOT/'checkpoints'/'sensorimotor_oracle_source_articulation_m0';training=SensorimotorTrainingConfig(batch_size=BATCH_SIZE,max_steps=MAX_STEPS,max_train_records=1000,max_validation_records=200,source_supervision_weight=1.0,control_contrastive_weight=1.0)
report=train_sensorimotor_vocoder(MANIFEST,RUN_DIR,training,config);print(json.dumps(report,ensure_ascii=False,indent=2))

In [ ]:
#@title 4. 控制分流严格门槛与试听
best=min(report['history'],key=lambda x:x['validation_spectral_loss'])
m0_pass=best['validation_source_relative_degradation']>.05 and best['validation_articulation_relative_degradation']>.05 and best['validation_mean_relative_degradation']>.05
print(json.dumps(best,ensure_ascii=False,indent=2));print('m0_pass =',m0_pass)
payload=torch.load(RUN_DIR/'validation_example.pt',map_location='cpu',weights_only=False);import torchaudio
torchaudio.save(str(RUN_DIR/'m0_input.wav'),payload['input'].unsqueeze(0),payload['sample_rate']);torchaudio.save(str(RUN_DIR/'m0_reconstruction.wav'),payload['reconstruction'].unsqueeze(0),payload['sample_rate'])
from IPython.display import Audio,display
print('输入：');display(Audio(filename=str(RUN_DIR/'m0_input.wav')));print('重建：');display(Audio(filename=str(RUN_DIR/'m0_reconstruction.wav')))
if not m0_pass:print('未通过：任一控制流打乱后的相对劣化必须超过 5%。')